# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [42]:
import re
#Copied from Lecture 9
def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"

#Copied from Lecture 9 but with some modifications to isolate HA/NA identifier and be barcode length agnostic
def read_barcode(seqread, upstream = "AGGCGGCCGC", ha_na_id_len=27):
    seq_str = str(seqread)
    seq_str = seq_str.upper()
    upstream = upstream.upper()
    upstream_seq = Bio.Seq.Seq(upstream)
    rev_comp_upstream = upstream_seq.reverse_complement()
    str_rev_comp_upstream = str(rev_comp_upstream)
    #Regex -- modified to isolate post immediate upstream sequence for HA/NA comparison
    search_pattern = re.compile(rf"^(?P<barcode>[ACGTN]*?){str_rev_comp_upstream}(?P<upstream_ID>[ACGTN]{{{ha_na_id_len}}})")

    barcode_match = search_pattern.search(seq_str)
    if barcode_match is None:
        return None
    else:
        barcode = str(Bio.Seq.Seq(barcode_match.group("barcode")).reverse_complement())
        upstream_ID = str(Bio.Seq.Seq(barcode_match.group("upstream_ID")).reverse_complement())
        return (barcode, upstream_ID)

#GCGGCCGCCT
test_cases = [
    # (read, expected result, label saying which requirement this checks)
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTTTTACCTACGTACCTACGTACCT", None, "ID sequence has < 27 bases"),
    ("TTTTTTTACGTTTTTTGCGGGCCT", None, "requirement 1: upstream sequence is wrong and ID sequence has < 27 bases"),
    ("YYYYAGCTYYYYAGCTGCGGCCGCCT", None, "requirement 3: downstream sequence contains invalid characters and ID sequence has < 27 bases"),
    ("AGCTAGCTGCGGCCGCCTTTTACCTACGTACCTACGTACCTACGT", ("AGCTAGCT", "ACGTAGGTACGTAGGTACGTAGGTAAA"), "Valid, non-16bp barcode"),
    ("AGCTAGAAACTGCGGCCGCCTTTTACCTACGTACCTACGTACCTACGT", ("AGTTTCTAGCT", "ACGTAGGTACGTAGGTACGTAGGTAAA"), "Valid, non-16bp barcode"),
    ("AGCTAGCTAGCTAGCTGAGGCCGCCT", None, "requirement 1: upstream sequence is wrong"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTTTTACCTACGTACCTACGTYCCTACGT", None, "Upstream ID sequence contains invalid characters"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTTTTACCTACGTACCTACGTCCTACGT", None, "Upstream ID sequence is too short"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTTTTACCTACGTACCTACGTTTCCTACGTTT", ("AAAAAAAAAAAAAAAA","CGTAGGAAACGTAGGTACGTAGGTAAA"), "positive"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTTTTACCTACGTACCTACGTTTCCTACGTGGGGTT", ("AAAAAAAAAAAAAAAA","CGTAGGAAACGTAGGTACGTAGGTAAA"), "positive")
]

run_tests(read_barcode, test_cases)

PASS  ID sequence has < 27 bases
PASS  requirement 1: upstream sequence is wrong and ID sequence has < 27 bases
PASS  requirement 3: downstream sequence contains invalid characters and ID sequence has < 27 bases
PASS  Valid, non-16bp barcode
PASS  Valid, non-16bp barcode
PASS  requirement 1: upstream sequence is wrong
PASS  Upstream ID sequence contains invalid characters
PASS  Upstream ID sequence is too short
PASS  positive
PASS  positive


'0 tests failed'

In [ ]:
import Bio.SeqIO
import Bio.Seq

#loading data
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

#Setting up data storage
HA_dict = {}
NA_dict = {}
failed_read_n = 0
failed_reads = []
unmatched_read_n = 0
unmatched_reads = []

for read in seqreads:

    output = read_barcode(read.seq)
    if(output is None):
        failed_read_n += 1
        failed_reads.append(read)
        continue
    else:
        barcode = output[0]
        upstream_ID = output[1]

    if(upstream_ID == "CACGATAGATAAATAATAGTGCACCAT"):
        NA_dict[barcode] = NA_dict.get(barcode, 0) + 1
    elif(upstream_ID == "CCGGATTTGCATATAATGATGCACCAT"):
        HA_dict[barcode] = HA_dict.get(barcode, 0) + 1
    else:
        unmatched_read_n += 1
        unmatched_reads.append(read)

print(f"{sum(HA_dict.values())} reads with a valid barcode map to HA.")
print(f"{sum(NA_dict.values())} reads with a valid barcode map to NA.")

5316 reads with a valid barcode map to HA.
3973 reads with a valid barcodemap to NA.


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [55]:
#Function to find invalid barcodes given a dictionary
def invalid_barcodes(dict, bclength = 16):
    #Finding the keys for a given dictionary (i.e, barcodes)
    barcodes = dict.keys()
    #Storing invalid barcodes
    invalid_barcodes = []
    #Iterating over each barcode; if the length is not equal to the specified length, add it to the invalid barcodes list
    for key in barcodes:
        if len(key) != bclength:
            invalid_barcodes.append(key)

    #Storing count for sequences w/o a valid barcode
    invalid_sequences = 0
    for key in invalid_barcodes:
        invalid_sequences += dict[key]

    return(invalid_barcodes, invalid_sequences)

HA_results = invalid_barcodes(HA_dict, 16)
NA_results = invalid_barcodes(NA_dict, 16)

print(f"There are {len(HA_results[0])} invalid barcodes in HA_dict corresponding with {HA_results[1]} sequences without a valid barcode.")
print(f"There are {len(NA_results[0])} invalid barcodes in NA_dict corresponding with {NA_results[1]} sequences without a valid barcode.")


There are 22 invalid barcodes in HA_dict corresponding with 70 sequences without a valid barcode.
There are 12 invalid barcodes in NA_dict corresponding with 63 sequences without a valid barcode.


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [56]:
#I bet there's a bespoke function to do this, but I'll just brute force one myself
def key_max(dict):
    keys = dict.keys()
    #storing max sequence
    max_seq = 0
    #iterating over keys
    for key in keys:
        if dict[key] > max_seq:
            max_seq = dict[key]

    return max_seq
    #return max(dict, key=dict.get) -- This was autosuggested; I knew it lol

HA_max_seq = key_max(HA_dict)
NA_max_seq = key_max(NA_dict)

print(f"The maximum number of sequences for a single HA barcode is {HA_max_seq}.")
print(f"The maximum number of sequences for a single NA barcode is {NA_max_seq}.")

The maximum number of sequences for a single HA barcode is 155.
The maximum number of sequences for a single NA barcode is 152.
